# Quantum Tunneling Laboratory
**Predict → calculate → test → explain → revise.** Browser Python (Pyodide); no widget extension is required.

Work in order. Change `E`, `V0`, and `a` in the parameter cell and rerun subsequent cells. Download your completed notebook: browser storage is local to this browser and origin. First launch needs internet to download Pyodide and packages.

Learning goals: explain transmission as current, derive boundary conditions, compare independent calculations, evaluate evidence, and report model limitations.

## 1. Predict before running
For E = 0.50 eV, V0 = 1.00 eV and a = 0.45 nm, predict whether T is zero, small, or near one. Predict the effect of doubling width. Record a reason and confidence (0–100%).

**Your prediction:** …

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "scipy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import linregress
print("Python ready:", sys.platform)

In [ ]:
"""Lossless equal-lead rectangular barrier; energies eV, positions nm.

Scattering states have unit incident amplitude, not unit integrated probability.
The API deliberately restricts energies to 0 < E < V0.
"""
import numpy as np

HBAR = 1.054571817e-34  # J s (CODATA rounded)
EV = 1.602176634e-19   # J / eV, exact
M_E = 9.1093837139e-31 # kg, CODATA 2022
NM = 1e-9
C = HBAR**2 / (2 * M_E * EV * NM**2)  # eV nm^2

def validate(E, V0, a):
    if not all(np.isfinite(v) for v in (E, V0, a)):
        raise ValueError('Parameters must be finite.')
    if not 0 < E < V0 or a < 0:
        raise ValueError('Require 0 < E < V0 and a >= 0 nm.')

def transmission(E=0.5, V0=1., a=0.45):
    """Exact transmission, evaluated in log space to avoid thick-barrier overflow."""
    validate(E, V0, a)
    if a == 0:
        return 1.
    z = np.sqrt((V0-E)/C)*a
    log_sinh = z + np.log(-np.expm1(-2*z)) - np.log(2.)
    log_term = 2*np.log(V0) + 2*log_sinh - np.log(4*E*(V0-E))
    return float(np.exp(-np.logaddexp(0., log_term)))

def amplitudes(E=0.5, V0=1., a=0.45):
    """Solve four continuity equations. Returns r, A, B, t.

    psi_left = exp(ikx) + r exp(-ikx)
    psi_inside = A exp(-kappa x) + B exp(kappa (x-a))
    psi_right = t exp(ik(x-a)). Both internal basis functions stay bounded.
    """
    validate(E, V0, a)
    k, q = np.sqrt(E/C), np.sqrt((V0-E)/C)
    d = np.exp(-q*a)
    matrix = np.array([[1,-1,-d,0],[-1j*k,q,-q*d,0],
                       [0,d,1,-1],[0,-q*d,q,-1j*k]], dtype=complex)
    rhs = np.array([-1,-1j*k,0,0], dtype=complex)
    return np.linalg.solve(matrix, rhs)

def wavefunction(x, E=0.5, V0=1., a=0.45):
    """psi and spatial derivative, x in nm; derivative per nm."""
    x = np.asarray(x, dtype=float)
    r,A,B,t = amplitudes(E,V0,a)
    k,q = np.sqrt(E/C),np.sqrt((V0-E)/C)
    p,d = np.empty(x.shape,complex),np.empty(x.shape,complex)
    left,inside,right = x<0,(x>=0)&(x<=a),x>a
    u,v = np.exp(1j*k*x[left]),r*np.exp(-1j*k*x[left])
    p[left],d[left] = u+v,1j*k*(u-v)
    u,v = A*np.exp(-q*x[inside]),B*np.exp(q*(x[inside]-a))
    p[inside],d[inside] = u+v,q*(-u+v)
    p[right] = t*np.exp(1j*k*(x[right]-a)); d[right] = 1j*k*p[right]
    return p,d

def numerical_transmission(E=.5,V0=1.,a=.45,rtol=1e-10):
    """Independent ODE integration backwards from a right-going exit state.

    For very opaque barriers cancellation limits accuracy; use modest widths
    for this pedagogical cross-check, not as an all-regime production solver.
    """
    from scipy.integrate import solve_ivp
    validate(E,V0,a)
    if a == 0: return 1.
    k = np.sqrt(E/C)
    sol = solve_ivp(lambda x,y:[y[1],(V0-E)/C*y[0]], (a,0),
                    np.array([1,1j*k]),rtol=rtol,atol=rtol*0.01)
    if not sol.success: raise RuntimeError(sol.message)
    p,d = sol.y[:,-1]
    incoming = (p+d/(1j*k))/2
    return float(1/abs(incoming)**2)

In [ ]:
# Edit these inputs, then Run All Below. Energies eV, length nm.
E = 0.50
V0 = 1.00
a = 0.45
validate(E, V0, a)
T = transmission(E,V0,a)
r,A,B,t = amplitudes(E,V0,a)
R = abs(r)**2
print(f"T={T:.9f}, R={R:.9f}, T+R={T+R:.12f}")
print(f"kappa={np.sqrt((V0-E)/C):.6f} nm^-1")
assert abs(T+R-1)<1e-10

## 2. Read the state carefully
Left: incident plus reflected waves. Right: outgoing only. The plotted real part is one phase choice. The stationary density is |psi|². A unit incident amplitude is a flux convention, not normalization over all space. Explain why ripples on the left do not imply electrons move up and down along the curve.

In [ ]:
x = np.linspace(-1, a+1, 1200)
psi, derivative = wavefunction(x,E,V0,a)
potential = np.where((x>=0)&(x<=a), V0, 0)
fig,axes=plt.subplots(2,1,figsize=(10,6),sharex=True)
axes[0].plot(x,potential,color="black",label="V (eV)")
axes[0].axhline(E,color="crimson",ls="--",label="E (eV)")
axes[0].set_ylabel("Energy (eV)");axes[0].legend()
axes[1].plot(x,psi.real,label="Re(psi), relative amplitude")
axes[1].plot(x,abs(psi)**2,label="|psi|², relative density")
axes[1].axvspan(0,a,color="gray",alpha=.15)
axes[1].set(xlabel="x (nm)",ylabel="Relative scale");axes[1].legend()
plt.tight_layout();plt.show()
# Current divided by incident current, proportional to Im(psi* dpsi/dx).
current = np.imag(psi.conj()*derivative)/np.sqrt(E/C)
assert np.max(abs(current-T))<1e-9
print("Constant normalized current:", current.min(), current.max())

In [ ]:
widths=np.linspace(0,1.2,200)
energies=np.linspace(.01,V0*.99,200)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].semilogy(widths,[transmission(E,V0,w) for w in widths])
axes[0].set(xlabel="Width (nm)",ylabel="T",title="Fixed energy and height")
axes[1].plot(energies,[transmission(e,V0,a) for e in energies])
axes[1].set(xlabel="Energy (eV)",ylabel="T",title="Fixed width and height")
plt.tight_layout();plt.show()
print("Exact T(2a)/T(a):",transmission(E,V0,2*a)/T)

## 3. Independent numerical calculation
Integrate the Schrödinger ODE backwards with psi(a)=1 and psi′(a)=ik. Resolve the incident amplitude at x=0 and compute its inverse squared magnitude. This does not reuse the closed-form transmission expression.

Predict what happens when the integration tolerance is loosened. For very opaque barriers this integration can suffer cancellation; investigate a moderate range first.

In [ ]:
for tolerance in [1e-4,1e-7,1e-10]:
    numerical=numerical_transmission(E,V0,a,rtol=tolerance)
    print(f"rtol={tolerance:g}: T_num={numerical:.10f}, error={abs(numerical-T):.3g}")
np.testing.assert_allclose(numerical,T,rtol=1e-7)
# A second check: boundary matching gives independently evaluated R and T.
np.testing.assert_allclose(abs(t)**2+abs(r)**2,1,atol=1e-10)

## 4. Find and repair a model error
The following intentionally incorrect function mixes electron volts and nanometers with SI constants. Execute it, explain the dimensional error, then implement a repaired version without calling `transmission` inside it. Compare your repaired result with the reference.

In [ ]:
def wrong_model(E_eV,V_eV,a_nm):
    wrong_kappa=np.sqrt(2*M_E*(V_eV-E_eV))/HBAR
    # Illustrates the wrong exponent without overflowing sinh.
    return np.exp(-2*wrong_kappa*a_nm)
print("Intentionally wrong output:",wrong_model(E,V0,a))
# TODO: define repaired_model(E_eV,V_eV,a_nm) and test its dimensions.

## 5. Design and critique an experiment
**Synthetic data below are generated, not measured device evidence.** Fit log(current) versus thickness. Decide what the slope can and cannot establish. Add a second candidate mechanism: an activated current `I0 * exp(-Ea/(k_B*T_temperature))`. Predict a temperature sweep for each mechanism and identify confounds.

In [ ]:
rng=np.random.default_rng(7)
measured_width=np.linspace(.25,.9,12)
# 10% log-normal instrumental/scatter illustration; uncertainty model is assumed.
current_data=np.array([transmission(.5,1,w) for w in measured_width])*np.exp(rng.normal(0,.10,12))
fit=linregress(measured_width,np.log(current_data))
print(f"log-slope={fit.slope:.3f} +/- {fit.stderr:.3f} nm^-1 (one standard error)")
plt.errorbar(measured_width,current_data,yerr=.1*current_data,fmt="o",label="synthetic data")
plt.plot(measured_width,np.exp(fit.intercept+fit.slope*measured_width),label="log-linear fit")
plt.yscale("log");plt.xlabel("Thickness (nm)");plt.ylabel("Relative current");plt.legend();plt.show()
# TODO: plot activated and weakly temperature-dependent candidate predictions.

## 6. Independent transfer task and submission
A new interface has an estimated effective mass `0.20*m_e` and a triangular rather than rectangular barrier under bias. You may not simply reuse the free-electron rectangular formula unchanged.

1. Derive how kappa changes if a constant effective mass were appropriate, and state the new interface matching assumptions.
2. Design a numerical solution for the triangular barrier and outline a convergence test.
3. Propose thickness, temperature, and bias measurements to compare direct tunneling, activation, and defect transport. Include instrument leakage and morphology controls.
4. Explain parameter uncertainty and identifiability: can one current measurement uniquely determine height and width?
5. Submit original code, a plot, a dimensional check, a boundary/current check, and a 300-word evidence argument. Revisit your initial prediction and confidence.

**Reflection / evidence argument:** …

Assessment rubric and pre/post/delayed tasks are in `docs/assessment.md`. Code MIT; original lesson text CC BY 4.0. Constants: CODATA 2022 free-electron mass, SI electron volt and reduced Planck constant as stated in the code.